# Voice under Stress - Analysis

import of all functions needed

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import scipy 
import math

#Stats
import statsmodels.api as sm
import statsmodels.formula.api as smf 
from statsmodels.stats.descriptivestats import sign_test
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.multivariate.manova import MANOVA 
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

#Figures
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams


In [ ]:
homepath=os.getcwd()
scripts_path=os.path.join(homepath, 'scripts')
sys.path.append(scripts_path)
import myml
import mystress
import myvoice

## Settings

In [ ]:
# Check version of Python==64
!python -c "import sys; print(sys.maxsize > 2**32)"

# random seed for reproducability
np.random.seed(42)

# show all columns of dataframes
pd.set_option('display.max_columns', None)

# set current dir to highest hierachy to add data path
os.chdir('/')
data_folder='./data'
sys.path.append(data_folder)
os.chdir(homepath)

# Main Code

### Functions for Figures

In [ ]:
def bar_cond(df, dv, group, subgroup, label):
    sns.set(context="paper", palette="bwr", style="ticks")
    ax=sns.violinplot(y=dv, x=group, palette={0: "white", 1: "#B7B7B7"}, hue=subgroup,
                      data=df)  
    plt.show()
    
def swarm_dualplot(df, dv, group, subgroup, label):
    """draws boxplots plus datapoints"""
    plt.figure(figsize=(8,4))
    sns.set(context="paper", style='whitegrid')
    ax = sns.boxplot(x=group, y=dv, data=df, hue='Cond', showfliers = False, color='grey')
    sns.swarmplot(x=group, y=dv, data=df, hue='Cond', dodge=True, ax=ax)
    plt.savefig('swarmplot_' + label + '.png', format='png', dpi=1200)
    plt.show()
    
def vpn_plots(df):
    """scatterplots with vpn-numbers as labels"""
    types = df.vpn
    y_coords = df_vpn.PANA_Delta_NA 
    x_coords = df_vpn.Cond #

    for i,type in enumerate(types):
        x = x_coords[i]
        y = y_coords[i]
        plt.scatter(x, y, marker='x', color='red')
        plt.text(x+0.01, y+0.01, type, fontsize=20)
    plt.show() 

### Statistical Functions

In [ ]:
def eta_squared(aov):
    aov['eta_sq'] = 'NaN'
    aov['eta_sq'] = aov[:-1]['sum_sq']/sum(aov['sum_sq'])
    return aov

def omega_squared(aov):
    mse = aov['sum_sq'][-1]/aov['df'][-1]
    aov['omega_sq'] = 'NaN'
    aov['omega_sq'] = (aov[:-1]['sum_sq']-(aov[:-1]['df']*mse))/(sum(aov['sum_sq'])+mse)
    return aov

### Step 0: Re-run Sound-Preprocessing

In [ ]:
#rerun_soundprep()

If you want to completely reanalyze all sound data, run: **rerun_soundprep()**

### Step 1: load data and calculate variables

In [ ]:
filename_librosa='./processed/features.csv'
filename_behavior=data_folder + '/raw/TSST_behavior.csv'
filename_praat='./processed/new_praat_results'
filename_opensmile='./processed/opensmile_features.csv'
filename_output='./processed/final_df.csv'

In [ ]:
df, librosa_featurenames, praat_featurenames, opensimle_feature=mystress.load_df(filename_opensmile, filename_praat, 
                                                 filename_librosa, filename_behavior, filename_output)

In [ ]:
df=mystress.calc_var(df)

In [ ]:
audio=praat_featurenames + list(opensimle_feature)+librosa_featurenames

### Step 2: Check the Dataset

In [ ]:
df[audio].describe(include = 'all')

[[ go back to the top ]](#Table-of-contents)


### Step 3: Stress Induction (Group Comparison)

In [ ]:
# entire sample age and bmi

col_of_interest = ['Age', 'BMI']
df[col_of_interest].describe(include='all')

In [ ]:
# split by condition for age and bmi
# Cond 0 = fTSST, Cond 1 = TSST

col_of_interest=['Age', 'BMI']
df.groupby('Cond')[col_of_interest].describe(include = 'all')

In [ ]:
# entire sample for sex

sex_total = pd.DataFrame([df['Sex'].agg(
    n='count',
    n_female=lambda s: (s == 1).sum(),
    pct_female=lambda s: 100 * (s == 1).mean()
)], index=['Total']).round({'pct_female': 1})
sex_total

In [ ]:
# split by condition for sex
# Cond 0 = fTSST, Cond 1 = TSST

sex_by_cond = df.groupby('Cond')['Sex'].agg(
    n='count',
    n_female=lambda s: (s == 1).sum(),
    pct_female=lambda s: 100 * (s == 1).mean()
).round({'pct_female': 1})
sex_by_cond

In [ ]:
# entire (female) sample for hormone status
# 1 = follicular, 2 = ovulation, 3 = luteal

cycle_by_cond = pd.crosstab(df['Cond'], df['cycle_phase'], dropna=False)
cycle_by_cond

cycle_by_cond_pct = (pd.crosstab(df['Cond'], df['cycle_phase'], normalize='index', dropna=False) * 100).round(1)
cycle_by_cond_pct

In [ ]:
# split by condition (females) for hormone status
# 1 = follicular, 2 = ovulation, 3 = luteal

cycle_by_cond_pct = (pd.crosstab(df['Cond'], df['cycle_phase'], normalize='index', dropna=False) * 100).round(1)
cycle_by_cond_pct

In [ ]:
col_of_interest=['Cortisol_MinMax', 'Cortisol_React', 'Delta_LNCort', 'LNco_AUCi', 'Delta_NA']
df.groupby('Cond')[col_of_interest].describe(include = 'all')

In [ ]:
swarm_dualplot(df, 'LNco_AUCi', 'Sex', 'Cond', 'LNco_AUCi')

In [ ]:
swarm_dualplot(df, 'Delta_LNCort', 'Sex', 'Cond', 'Delta_LNCort')

In [ ]:
swarm_dualplot(df, 'Cortisol_React', 'Sex', 'Cond', 'Cortisol_MinMax')

In [ ]:
formula = 'Cortisol_React ~ C(Cond) + C(Sex)+ C(Cond): C(Sex)'
model = ols(formula, df).fit()
print (model.summary())

In [ ]:
# Hier für sAA
formula = 'sAA_React ~ C(Cond) + C(Sex) + C(Cond):C(Sex)'
model = ols(formula, df).fit()
print(model.summary())

In [ ]:
# Hier für PANAS - NEGATIVE
formula = 'PANA_Delta_NA ~ C(Cond) + C(Sex) + C(Cond):C(Sex)'
model = ols(formula, df).fit()
print(model.summary())

In [ ]:
# Hier für PANAS - POSITIVE
formula = 'PANA_Delta_PA ~ C(Cond) + C(Sex) + C(Cond):C(Sex)'
model = ols(formula, df).fit()
print(model.summary())

In [ ]:
# test for heteroskedacity

formulas = {
    "Cortisol_React": "Cortisol_React ~ C(Cond) + C(Sex) + C(Cond):C(Sex)",
    "sAA_React": "sAA_React ~ C(Cond) + C(Sex) + C(Cond):C(Sex)",
    "PANA_Delta_NA": "PANA_Delta_NA ~ C(Cond) + C(Sex) + C(Cond):C(Sex)",
    "PANA_Delta_PA": "PANA_Delta_PA ~ C(Cond) + C(Sex) + C(Cond):C(Sex)",
}

rows = []
for name, f in formulas.items():
    res = smf.ols(f, data=df).fit()
    bp = het_breuschpagan(res.resid, res.model.exog)
    wt = het_white(res.resid, res.model.exog)
    rows.append({
        "model": name,
        "BP_p_LM": bp[1],
        "BP_p_F":  bp[3],
        "White_p_LM": wt[1],
        "White_p_F":  wt[3],
    })

het_tests = pd.DataFrame(rows).set_index("model")
het_tests

In [ ]:
# since heteroskedacity was found for cortisol and sAA is trending there too, we compute hc3 robust errors

formulas = {
    "Cortisol_React": "Cortisol_React ~ C(Cond) + C(Sex) + C(Cond):C(Sex)",
    "sAA_React": "sAA_React ~ C(Cond) + C(Sex) + C(Cond):C(Sex)",
    "PANA_Delta_NA": "PANA_Delta_NA ~ C(Cond) + C(Sex) + C(Cond):C(Sex)",
    "PANA_Delta_PA": "PANA_Delta_PA ~ C(Cond) + C(Sex) + C(Cond):C(Sex)",
}

res = smf.ols(formulas["Cortisol_React"], data=df).fit()
res_hc3 = res.get_robustcov_results(cov_type="HC3")
res_hc3.summary()

In [ ]:
f = "Cortisol_React ~ C(Cond) + C(Sex) + C(Cond):C(Sex)"
m = smf.ols(f, data=df).fit()

m_hc3 = m.get_robustcov_results(cov_type="HC3")
p_hc3 = pd.Series(m_hc3.pvalues, index=m.params.index)

term = "C(Cond)[T.1]"
print("Classical p(Cond):", m.pvalues[term])
print("HC3 p(Cond):      ", p_hc3[term])

In [ ]:
res = smf.ols(formulas["sAA_React"], data=df).fit()
res_hc3 = res.get_robustcov_results(cov_type="HC3")
res_hc3.summary()

In [ ]:
f = "sAA_React ~ C(Cond) + C(Sex) + C(Cond):C(Sex)"
m = smf.ols(f, data=df).fit()

m_hc3 = m.get_robustcov_results(cov_type="HC3")
p_hc3 = pd.Series(m_hc3.pvalues, index=m.params.index)

term = "C(Cond)[T.1]"
print("Classical p(Cond):", m.pvalues[term])
print("HC3 p(Cond):      ", p_hc3[term])

In [ ]:
res = smf.ols(formulas["PANA_Delta_NA"], data=df).fit()
res_hc3 = res.get_robustcov_results(cov_type="HC3")
res_hc3.summary()

In [ ]:
f = "PANA_Delta_NA ~ C(Cond) + C(Sex) + C(Cond):C(Sex)"
m = smf.ols(f, data=df).fit()

m_hc3 = m.get_robustcov_results(cov_type="HC3")
p_hc3 = pd.Series(m_hc3.pvalues, index=m.params.index)

term = "C(Cond)[T.1]"
print("Classical p(Cond):", m.pvalues[term])
print("HC3 p(Cond):      ", p_hc3[term])

In [ ]:
res = smf.ols(formulas["PANA_Delta_PA"], data=df).fit()
res_hc3 = res.get_robustcov_results(cov_type="HC3")
res_hc3.summary()

In [ ]:
f = "PANA_Delta_PA ~ C(Cond) + C(Sex) + C(Cond):C(Sex)"
m = smf.ols(f, data=df).fit()

m_hc3 = m.get_robustcov_results(cov_type="HC3")
p_hc3 = pd.Series(m_hc3.pvalues, index=m.params.index)

term = "C(Cond)[T.1]"
print("Classical p(Cond):", m.pvalues[term])
print("HC3 p(Cond):      ", p_hc3[term])

### Time course (cortisol/sAA) significance testing ###

In [ ]:
# for cortisol

long = pd.melt(
    df, id_vars=["vpn","Cond","Sex"],
    value_vars=["LNCortisol_BL","LNCortisol_post1","LNCortisol_post20"],
    var_name="Time", value_name="LNCortisol"
)

long["Time"] = long["Time"].map({
    "LNCortisol_BL":"Baseline",
    "LNCortisol_post1":"post1",
    "LNCortisol_post20":"post20"
})

m = smf.mixedlm("LNCortisol ~ C(Time)*C(Cond)*C(Sex)", long, groups=long["vpn"])
res = m.fit(reml=False)
print(res.summary())

In [ ]:
# for sAA

long = pd.melt(
    df, id_vars=["vpn","Cond","Sex"],
    value_vars=["LNsAA_BL","LNsAA_post1","LNsAA_post20"],
    var_name="Time", value_name="LNsAA"
)

long["Time"] = long["Time"].map({
    "LNsAA_BL":"Baseline",
    "LNsAA_post1":"post1",
    "LNsAA_post20":"post20"
})

m = smf.mixedlm("LNsAA ~ C(Time)*C(Cond)*C(Sex)", long, groups=long["vpn"])
res = m.fit(reml=False)
print(res.summary())

In [ ]:
# graph for the time course of cortisol and sAA (LN)

def _make_long(df, value_cols, value_name):
    long = pd.melt(
        df,
        id_vars=["vpn", "Cond", "Sex"],
        value_vars=value_cols,
        var_name="Time",
        value_name=value_name
    )

    time_map = {
        value_cols[0]: "Baseline",
        value_cols[1]: "post1",
        value_cols[2]: "post20",
    }
    long["Time"] = long["Time"].map(time_map)
    long["Time"] = pd.Categorical(long["Time"], categories=["Baseline", "post1", "post20"], ordered=True)
    return long

def _summarize(long, ycol):
    g = long.groupby(["Sex", "Cond", "Time"])[ycol]
    summary = g.agg(mean="mean", sd="std", n="count").reset_index()
    summary["se"] = summary["sd"] / np.sqrt(summary["n"])
    return summary

def plot_timecourse(summary, y_label, title_prefix=""):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

    for ax, sex_val in zip(axes, [0, 1]):
        sub = summary[summary["Sex"] == sex_val].copy()

        for cond_val, cond_name in [(0, "fTSST (0)"), (1, "TSST (1)")]:
            s = sub[sub["Cond"] == cond_val].sort_values("Time")
            ax.errorbar(
                s["Time"].astype(str),
                s["mean"],
                yerr=s["se"],
                marker="o",
                capsize=3,
                label=cond_name
            )

        ax.set_title(f"{title_prefix}Sex={sex_val}")
        ax.set_xlabel("Time")
        ax.grid(True, axis="y", alpha=0.3)

    axes[0].set_ylabel(y_label)
    axes[1].legend(loc="best")

    fig.tight_layout()
    plt.show()

# --- 1) Log cortisol ---
cort_long = _make_long(df, ["LNCortisol_BL", "LNCortisol_post1", "LNCortisol_post20"], "LNCortisol")
cort_sum = _summarize(cort_long, "LNCortisol")
plot_timecourse(cort_sum, y_label="Log cortisol", title_prefix="Cortisol: ")

# --- 2) Log sAA ---
saa_long = _make_long(df, ["LNsAA_BL", "LNsAA_post1", "LNsAA_post20"], "LNsAA")
saa_sum = _summarize(saa_long, "LNsAA")
plot_timecourse(saa_sum, y_label="Log sAA", title_prefix="sAA: ")

In [ ]:
# graphs for the time course of cortisol and sAA (LN)

def _make_long(df, value_cols, value_name):
    long = pd.melt(
        df,
        id_vars=["vpn", "Cond", "Sex"],
        value_vars=value_cols,
        var_name="Time",
        value_name=value_name
    )

    time_map = {
        value_cols[0]: "Baseline",
        value_cols[1]: "post1",
        value_cols[2]: "post20",
    }
    long["Time"] = long["Time"].map(time_map)
    long["Time"] = pd.Categorical(long["Time"], categories=["Baseline", "post1", "post20"], ordered=True)
    return long

def _summarize(long, ycol):
    g = long.groupby(["Sex", "Cond", "Time"])[ycol]
    summary = g.agg(mean="mean", sd="std", n="count").reset_index()
    summary["se"] = summary["sd"] / np.sqrt(summary["n"])
    return summary

def _summarize_pooled(long, ycol):
    g = long.groupby(["Cond", "Time"])[ycol]
    summary = g.agg(mean="mean", sd="std", n="count").reset_index()
    summary["se"] = summary["sd"] / np.sqrt(summary["n"])
    return summary

def plot_timecourse(summary, y_label, title_prefix=""):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

    sex_label = {0: "Male", 1: "Female"}

    for ax, sex_val in zip(axes, [0, 1]):
        sub = summary[summary["Sex"] == sex_val].copy()

        for cond_val, cond_name in [(0, "f-TSST"), (1, "TSST")]:
            s = sub[sub["Cond"] == cond_val].sort_values("Time")
            ax.errorbar(
                s["Time"].astype(str),
                s["mean"],
                yerr=s["se"],
                marker="o",
                capsize=3,
                label=cond_name
            )

        ax.set_title(f"{title_prefix}{sex_label[sex_val]}")
        ax.set_xlabel("Time")
        ax.grid(True, axis="y", alpha=0.3)

    axes[0].set_ylabel(y_label)
    axes[1].legend(loc="best")

    fig.tight_layout()
    plt.show()

def plot_timecourse_pooled(summary, y_label, title_prefix=""):
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))

    for cond_val, cond_name in [(0, "f-TSST"), (1, "TSST")]:
        s = summary[summary["Cond"] == cond_val].sort_values("Time")
        ax.errorbar(
            s["Time"].astype(str),
            s["mean"],
            yerr=s["se"],
            marker="o",
            capsize=3,
            label=cond_name
        )

    ax.set_title(f"{title_prefix}All participants")
    ax.set_xlabel("Time")
    ax.set_ylabel(y_label)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="best")

    fig.tight_layout()
    plt.show()

# Log cortisol
cort_long = _make_long(df, ["LNCortisol_BL", "LNCortisol_post1", "LNCortisol_post20"], "LNCortisol")
cort_sum = _summarize(cort_long, "LNCortisol")
plot_timecourse(cort_sum, y_label="Log cortisol", title_prefix="Cortisol: ")

cort_sum_pooled = _summarize_pooled(cort_long, "LNCortisol")
plot_timecourse_pooled(cort_sum_pooled, y_label="Log cortisol", title_prefix="Cortisol: ")

# Log sAA
saa_long = _make_long(df, ["LNsAA_BL", "LNsAA_post1", "LNsAA_post20"], "LNsAA")
saa_sum = _summarize(saa_long, "LNsAA")
plot_timecourse(saa_sum, y_label="Log sAA", title_prefix="sAA: ")

saa_sum_pooled = _summarize_pooled(saa_long, "LNsAA")
plot_timecourse_pooled(saa_sum_pooled, y_label="Log sAA", title_prefix="sAA: ")

In [ ]:
TIME_ORDER = ["Baseline", "post1", "post20"]

def _make_long(df, value_cols, value_name):
    long = pd.melt(
        df,
        id_vars=["vpn", "Cond", "Sex"],
        value_vars=value_cols,
        var_name="Time",
        value_name=value_name
    )
    time_map = {
        value_cols[0]: "Baseline",
        value_cols[1]: "post1",
        value_cols[2]: "post20",
    }
    long["Time"] = long["Time"].map(time_map)
    long["Time"] = pd.Categorical(long["Time"], categories=TIME_ORDER, ordered=True)
    return long

def _summarize_pooled(long, ycol):
    g = long.groupby(["Cond", "Time"])[ycol]
    summary = g.agg(mean="mean", sd="std", n="count").reset_index()
    summary["se"] = summary["sd"] / np.sqrt(summary["n"])
    return summary

def _plot_mean_se(ax, summary, y_label, title):
    for cond_val, cond_name in [(1, "TSST"), (0, "f-TSST")]: 
        s = summary[summary["Cond"] == cond_val].sort_values("Time")
        ax.errorbar(
            s["Time"].astype(str),
            s["mean"],
            yerr=s["se"],
            marker="o",
            capsize=3,
            label=cond_name
        )
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel(y_label)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="best")

def plot_timecourse(df):
    cort_long = _make_long(df, ["LNCortisol_BL","LNCortisol_post1","LNCortisol_post20"], "LNCortisol")
    saa_long  = _make_long(df, ["LNsAA_BL","LNsAA_post1","LNsAA_post20"], "LNsAA")

    cort_sum = _summarize_pooled(cort_long, "LNCortisol")
    saa_sum  = _summarize_pooled(saa_long,  "LNsAA")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
    _plot_mean_se(axes[0], cort_sum, "Log cortisol", "Cortisol (log): TSST vs f-TSST")
    _plot_mean_se(axes[1], saa_sum,  "Log sAA",      "sAA (log): TSST vs f-TSST")
    fig.tight_layout()
    
    return fig

fig = plot_timecourse(df)
fig.savefig("biomarker_timecourse.png", dpi=300, bbox_inches="tight")
fig.savefig("biomarker_timecourse.pdf", bbox_inches="tight")
plt.show()

# Voice Analysis (Group Comparison)

### Descriptive Values

In [ ]:
col_of_interest=['agg_f0mean', 'meanF0Hz', 'stdevF0Hz', 'energy', 
                 'localJitter', 'localShimmer', 'HNR', 'median_pitch']

print (df.groupby(['Sex', 'Cond'])[col_of_interest].count())
print (df.groupby(['Sex', 'Cond'])[col_of_interest].mean())
print (df.groupby(['Sex', 'Cond'])[col_of_interest].std())

### Statistical Tests of Group Differences (Voice Features)

Stressed (1) vs. Friendly (0) and Female (1) vs. Male (0)

First Check: MANOVA -> Are there any group differences?

In [ ]:
maov = MANOVA.from_formula('meanF0Hz + stdevF0Hz + HNR + localShimmer + localJitter + energy ~ Cond + Sex + Cond:Sex', data=df)
print(maov.mv_test())

## Fundamental Frequency: Mean & Std

In [ ]:
formula = 'meanF0Hz ~  C(Sex) + C(Cond)'
model = ols(formula, df).fit()
aov_table = anova_lm(model, typ=2)
#https://www.marsja.se/three-ways-to-carry-out-2-way-anova-with-python/

eta_squared(aov_table)
omega_squared(aov_table)
print(aov_table.round(4))

In [ ]:
formula = 'meanF0Hz ~ C(Cond) + C(Sex)'
model = ols(formula, df).fit()
print (model.summary())

In [ ]:
swarm_dualplot(df, 'meanF0Hz', 'Sex', 'Cond', 'meanF0Hz')

In [ ]:
formula = 'stdevF0Hz ~ C(Cond) + C(Sex)'
model = ols(formula, df).fit()
print (model.summary())

In [ ]:
swarm_dualplot(df, 'stdevF0Hz', 'Sex', 'Cond', 'meanF0Hz')

## Microvariation: Shimmer & Jitter

In [ ]:
formula = 'localJitter ~  C(Cond)* C(Sex)'
model = ols(formula, df).fit()
print (model.summary())

In [ ]:
swarm_dualplot(df, 'localJitter', 'Sex', 'Cond', 'localJitter')

In [ ]:
formula = 'localShimmer ~  C(Cond) * C(Sex)'
model = ols(formula, df).fit()
print (model.summary())

In [ ]:
swarm_dualplot(df, 'localShimmer', 'Sex', 'Cond', 'localShimmer')

## Energy

In [ ]:
import mystats
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure()
sns.set(style="whitegrid")
ax = sns.boxplot(x="Cond", y="energy", data=df, showfliers = False)
sns.swarmplot(x="Cond", y="energy", data=df, color=".01", ax=ax)
plt.show()



In [ ]:
formula = 'energy ~ C(Cond)+C(Sex)'
model = ols(formula, df).fit()
print (model.summary())

## Harmony-To-Noise Ratio (HNR)

In [ ]:
formula = 'HNR ~ C(Sex)+C(Cond)'
model = ols(formula, df).fit()
print (model.summary())

In [ ]:
formula = 'HNR ~ C(Sex)+C(Cond)+Delta_LNCort'
model = ols(formula, df).fit()
print (model.summary())

In [ ]:
swarm_dualplot(df, 'HNR', 'Sex', 'Cond', 'HNR')

# Time of Speaking (Audio lengths)

In [ ]:
# descriptives for length distributions before and after processing/cutting

In [ ]:
import mystats
formula = 'length ~ C(Cond)+C(Sex)'

model = ols(formula, df).fit()
print (model.summary())
mystats.two_ind_sample_tests(df[df.Cond==0], df[df.Cond==1], 'length')

In [ ]:
# Mean and SD of length per condition for PROCESSED audio
df.groupby("Cond")["length"].agg(["count", "mean", "std", "min", "median", "max"])

In [ ]:
# Convert length into seconds/minutes
sr = 22050        # default librosa
hop_length = 512  # default librosa

hop_s = hop_length / sr

# Original (frames)
len_frames = df.groupby("Cond")["length"].agg(["count", "mean", "std", "min", "median", "max"])

# Converted to seconds
len_seconds = len_frames.copy()
for c in ["mean", "std", "min", "median", "max"]:
    len_seconds[c] = len_seconds[c] * hop_s

# Converted to minutes
len_minutes = len_seconds.copy()
for c in ["mean", "std", "min", "median", "max"]:
    len_minutes[c] = len_minutes[c] / 60


len_frames.index.name = "Cond"
len_seconds.index.name = "Cond"
len_minutes.index.name = "Cond"

print("Length in frames (as stored):")
display(len_frames)

print(f"\nLength converted to seconds (sr={sr}, hop_length={hop_length}, hop_s={hop_s:.6f}):")
display(len_seconds)

print("\nLength converted to minutes:")
display(len_minutes)

In [ ]:
plt.figure()
sns.set(style="whitegrid")

ax = sns.boxplot(x="Cond", y="length", data=df, showfliers = False)
sns.swarmplot(x="Cond", y="length", data=df, color=".01", ax=ax)

plt.show()

https://github.com/drfeinberg/PraatScripts

In [ ]:
# für die Stressgruppe (?): Zusammenhänge von Cortisolanstieg und Stimmung mit den Stimmparametern
# Achtung: für Frequenzwerte muss nach Männern und Frauen getrennt werden
rcParams['figure.figsize'] = 10, 8
audio_gender=['meanF0Hz', 'stdevF0Hz', 'engergy']
audio_small=['localShimmer', 'localJitter', 'Cortisol_React', 'Delta_NA']
plt.matshow(df[audio_small].corr())
plt.xticks(np.arange(df[audio_small].shape[1]), df[audio_small].columns)
plt.yticks(np.arange(df[audio_small].shape[1]), df[audio_small].columns)
#ax = sns.heatmap(df_vpn[audio_small].corr(), vmin=0, vmax=1)
plt.colorbar()
plt.show()

Christine --

In [ ]:
df.groupby(["Cond", "Sex"])["length"].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    p10=lambda x: np.percentile(x, 10),
    p25=lambda x: np.percentile(x, 25),
    p75=lambda x: np.percentile(x, 75),
    p90=lambda x: np.percentile(x, 90),
    max="max"
)

sns.histplot(df["length"], kde=True)
plt.title("Distribution of Speaking Length")
plt.xlabel("Length (seconds)")
plt.ylabel("Count")
plt.show()

sns.histplot(data=df, x="length", hue="Cond", kde=True, element="step")
plt.title("Length Distribution by Condition")
plt.show()

sns.ecdfplot(data=df, x="length", hue="Cond")
plt.title("Cumulative Distribution of Length")
plt.show()

### Graphics

In [ ]:
swarm_dualplot(df, 'HNR', 'Sex', 'Cond', 'Figure1')

In [ ]:
plt.figure(figsize=(8,4))
sns.set(style="whitegrid")
plt.hist(df.energy)
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.set(style="whitegrid")
#plt.plot(df.groupby('Cond').mean()[['cort_baseline', 'cort_01_min', 'cort_20_min']].T)
plt.plot(
    df.groupby('Cond')[['cort_baseline', 'cort_01_min', 'cort_20_min']]
      .mean()
      .T
)

plt.show()

In [ ]:
#df.groupby('Cond').mean()[['cort_baseline', 'cort_01_min', 'cort_20_min']]
df.groupby("Cond")[["cort_baseline", "cort_01_min", "cort_20_min"]].mean()
